In [1]:
print("Hello . You can do it !")

Hello . You can do it !


In [2]:
import torch
import torch.nn as nn
import math
import h5py
from torch.utils.data import Dataset, DataLoader

In [3]:
path = r"/kaggle/input/datasets/lucasxyz/hcd-data"

In [4]:
path_train = path + "/traintest_hcd.hdf5"
path_test  = path + "/holdout_hcd.hdf5"

In [5]:
import h5py

In [6]:
f = h5py.File(path_train, "r")
for key in list(f.keys()):
    print(key)


collision_energy
collision_energy_aligned
collision_energy_aligned_normed
intensities_raw
masses_pred
masses_raw
method
precursor_charge_onehot
rawfile
reverse
scan_number
score
sequence_integer
sequence_onehot


In [7]:
intensity = f["intensities_raw"][0]
intensity.reshape(29,6)

array([[ 0.03333019,  0.        ,  0.        ,  0.00833965,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.39771285,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.12638215,  0.        ,
         0.        ],
       [ 0.00881359,  0.        ,  0.        ,  0.0085394 ,  0.        ,
         0.        ],
       [ 0.02134586,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.01633287,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.01483933,  0.        ,  0.02334369,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.00457667,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.00765157,  0.        ,  0.

In [8]:
print(intensity.max())
print(intensity.min())


1.0
-1.0


In [9]:
sequence = f["sequence_integer"][0]
sequence

array([19,  4, 18, 20, 13, 18, 12, 16, 17,  5, 15, 18, 10, 18, 16,  6,  1,
       17, 15,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [10]:
masses_raw = f["masses_raw"][0]
len(masses_raw)



174

In [11]:
# import h5py
# import torch
# from tqdm import tqdm

# hdf5_path = path_train
# save_path = "/kaggle/working/targets.pt"

# chunk_size = 50000  # 50k mẫu/lần

# with h5py.File(hdf5_path, "r") as data:

#     N = len(data["sequence_integer"])
#     targets = torch.zeros(N, 29, 6, dtype=torch.float32)

#     for start in tqdm(range(0, N, chunk_size)):

#         end = min(start + chunk_size, N)

#         intensities = torch.tensor(
#             data["intensities_raw"][start:end],
#             dtype=torch.float32
#         )

#         seqs = torch.tensor(
#             data["sequence_integer"][start:end]
#         )

#         lengths = (seqs != 0).sum(dim=1)

#         frag_range = torch.arange(1, 30).unsqueeze(0).repeat(end-start, 1)

#         valid_mask = frag_range < lengths.unsqueeze(1)

#         idx_base_b = frag_range - 1
#         idx_b = torch.stack([
#             0 * 29 + idx_base_b,
#             1 * 29 + idx_base_b,
#             2 * 29 + idx_base_b
#         ], dim=2).clamp(min=0)

#         y_pos = lengths.unsqueeze(1) - frag_range
#         idx_base_y = y_pos - 1

#         idx_y = torch.stack([
#             87 + 0 * 29 + idx_base_y,
#             87 + 1 * 29 + idx_base_y,
#             87 + 2 * 29 + idx_base_y
#         ], dim=2).clamp(min=0)

#         expanded = intensities.unsqueeze(1).expand(-1, 29, -1)

#         b_vals = torch.gather(expanded, 2, idx_b)
#         y_vals = torch.gather(expanded, 2, idx_y)

#         chunk_target = torch.zeros(end-start, 29, 6)
#         chunk_target[:, :, :3] = b_vals
#         chunk_target[:, :, 3:] = y_vals
#         chunk_target[~valid_mask] = 0

#         targets[start:end] = chunk_target

# torch.save(targets, save_path)

In [12]:
import h5py
import torch
from tqdm import tqdm

hdf5_path = path_train
save_path = "/kaggle/working/targets.pt"

chunk_size = 50000

with h5py.File(hdf5_path, "r") as data:

    N = len(data["sequence_integer"])
    targets = torch.zeros(N, 29, 6, dtype=torch.float32)

    for start in tqdm(range(0, N, chunk_size)):

        end = min(start + chunk_size, N)

        intensities = torch.tensor(
            data["intensities_raw"][start:end],
            dtype=torch.float32
        )

        seqs = torch.tensor(
            data["sequence_integer"][start:end]
        )

        lengths = (seqs != 0).sum(dim=1)

        frag_range = torch.arange(1, 30).unsqueeze(0).repeat(end-start, 1)

        valid_mask = frag_range < lengths.unsqueeze(1)

        # reshape 174 → (29,6)
        chunk_target = intensities.view(end-start, 29, 6)

        chunk_target[chunk_target < 0] = 0

        # mask fragment không hợp lệ
        chunk_target[~valid_mask] = 0

        targets[start:end] = chunk_target

torch.save(targets, save_path)

100%|██████████| 136/136 [01:18<00:00,  1.74it/s]


In [13]:
# import torch

# def prosit_to_pdeep_fast(intensity_174, seq_len, max_frag=29):
#     """
#     intensity_174: (174,)
#     seq_len: số amino acid thật (không tính padding)
#     return: (29, 6)  -- đã pad sẵn
#     """

#     intensity = torch.as_tensor(intensity_174, dtype=torch.float32)

#     # tạo target cố định 29 cleavage
#     target = torch.zeros(29, 6, dtype=torch.float32)

#     if seq_len <= 1:
#         return target

#     L = seq_len
#     frag_range = torch.arange(1, L)          # 1 .. L-1
#     L_minus_1 = L - 1

#     # ---------- b ions ----------
#     valid_b = frag_range <= max_frag
#     idx_base_b = frag_range - 1

#     idx_b = torch.stack([
#         0 * 29 + idx_base_b,
#         1 * 29 + idx_base_b,
#         2 * 29 + idx_base_b
#     ], dim=1)  # shape (L-1, 3)

#     target[:L_minus_1, :3][valid_b] = intensity[idx_b[valid_b]]

#     # ---------- y ions ----------
#     y_pos = L - frag_range
#     valid_y = y_pos <= max_frag
#     idx_base_y = y_pos - 1

#     idx_y = torch.stack([
#         87 + 0 * 29 + idx_base_y,
#         87 + 1 * 29 + idx_base_y,
#         87 + 2 * 29 + idx_base_y
#     ], dim=1)  # shape (L-1, 3)

#     target[:L_minus_1, 3:][valid_y] = intensity[idx_y[valid_y]]

#     return target

In [14]:
import torch

def prosit_to_pdeep_fast(intensity_174, seq_len, max_frag=29):
    """
    intensity_174: (174,)
    seq_len: số amino acid thật
    return: (29,6)
    """

    intensity = torch.as_tensor(intensity_174, dtype=torch.float32)

    # reshape trực tiếp
    target = intensity.view(29, 6).clone()

    if seq_len <= 1:
        return torch.zeros(29,6)

    L = seq_len
    valid_len = min(L - 1, max_frag)
    
    target[target < 0] = 0
    
    # mask fragment không tồn tại
    target[valid_len:] = 0

    return target

In [15]:
import h5py
import torch
from tqdm import tqdm

hdf5_path = path_train

with h5py.File(hdf5_path, "r") as data:

    print("Loading sequences...")
    all_seqs = torch.tensor(
        data["sequence_integer"][:],
        dtype=torch.long
    )

    print("Loading charge...")
    charge_onehot = torch.tensor(
        data["precursor_charge_onehot"][:]
    )
    all_charge = (
        charge_onehot.argmax(dim=1) * 0.1 + 0.1
    ).unsqueeze(1).float()

    print("Loading NCE...")
    all_nce = torch.tensor(
        data["collision_energy_aligned_normed"][:],
        dtype=torch.float32
    )

print("Saving...")
torch.save(all_seqs, "/kaggle/working/seq.pt")
torch.save(all_charge, "/kaggle/working/charge.pt")
torch.save(all_nce, "/kaggle/working/nce.pt")

print("Done.")

Loading sequences...
Loading charge...
Loading NCE...
Saving...
Done.


In [16]:
import h5py
import torch
from torch.utils.data import Dataset

class PDeepDataset(Dataset):
    def __init__(self):
        self.seqs = torch.load("/kaggle/working/seq.pt")
        self.charge = torch.load("/kaggle/working/charge.pt")
        self.nce = torch.load("/kaggle/working/nce.pt")
        self.targets = torch.load("/kaggle/working/targets.pt")

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, idx):

        seq = self.seqs[idx]
        charge = self.charge[idx]
        nce = self.nce[idx]
        target = self.targets[idx]
        mask = (seq != 0)[:-1]
        last = mask.sum() - 1
        mask[last] = False
        mask = mask.float()

        return seq, charge, nce, target, mask

In [17]:
dataset = PDeepDataset()
len(dataset)

6787933

In [18]:
seq, charge, nce, target , mask = dataset[1]

print("Sequence shape:", seq.shape)
print("Charge:", charge)
print("NCE:", nce)
print("Target shape:", target.shape)

Sequence shape: torch.Size([30])
Charge: tensor([0.2000])
NCE: tensor([0.2233])
Target shape: torch.Size([29, 6])


In [19]:
seq[:]

tensor([10, 13, 12,  5, 10,  4, 10,  8,  9, 21,  8, 15,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [20]:
mask[:]


tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [21]:
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [22]:
seq, charge, nce, target, mask = next(iter(loader))

print("seq shape:", seq.shape)
print("charge shape:", charge.shape)
print("nce shape:", nce.shape)
print("target shape:", target.shape)
print("mask shape:", mask.shape)

seq shape: torch.Size([8, 30])
charge shape: torch.Size([8, 1])
nce shape: torch.Size([8, 1])
target shape: torch.Size([8, 29, 6])
mask shape: torch.Size([8, 29])


In [23]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(max_len) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [24]:
class MetaEmbedding(nn.Module):
    def __init__(self, meta_dim=8):
        super().__init__()
        self.linear = nn.Linear(1, meta_dim - 1)

    def forward(self, charge, nce):
        meta = self.linear(nce)
        meta = torch.cat([meta, charge], dim=1)
        return meta

In [25]:
class ModelMS2Transformer(nn.Module):
    def __init__(
        self,
        aa_vocab_size,
        mod_dim,
        num_frag_types,
        num_modloss_types=0,
        hidden=256,
        dropout=0.1,
        nlayers=4,
        mask_modloss=True
    ):
        super().__init__()

        self.num_modloss = num_modloss_types
        self.num_non_modloss = num_frag_types - num_modloss_types
        self.mask_modloss = mask_modloss

        meta_dim = 8

        # ==== Input embedding ====
        self.aa_embedding = nn.Embedding(
            aa_vocab_size,
            hidden - meta_dim - mod_dim
        )

        if mod_dim > 0:
            self.mod_linear = nn.Linear(mod_dim, mod_dim)
        else:
            self.mod_linear = None

        self.pos_encoder = PositionalEncoding(hidden - meta_dim)

        self.meta_embedding = MetaEmbedding(
            meta_dim=meta_dim
        )

        # ==== Transformer backbone ====
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=nlayers
        )

        self.dropout = nn.Dropout(dropout)

        # ==== Main head ====
        self.output_head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.PReLU(),
            nn.Linear(64, self.num_non_modloss)
        )

        # ==== Modloss branch ====
        if num_modloss_types > 0:
            modloss_layer = nn.TransformerEncoderLayer(
                d_model=hidden,
                nhead=8,
                dim_feedforward=hidden * 4,
                dropout=dropout,
                batch_first=True
            )

            self.modloss_transformer = nn.TransformerEncoder(
                modloss_layer,
                num_layers=1
            )

            self.modloss_head = nn.Sequential(
                nn.Linear(hidden, 64),
                nn.PReLU(),
                nn.Linear(64, num_modloss_types)
            )
        else:
            self.modloss_transformer = None
            self.modloss_head = None
    def forward(self, aa_idx, mod_x, charge, nce):
        B, L = aa_idx.shape

        # ==== AA + Mod embedding ====
        aa_emb = self.aa_embedding(aa_idx)
        if self.mod_linear is not None:
            mod_emb = self.mod_linear(mod_x)
            seq_x = torch.cat([aa_emb, mod_emb], dim=2)
        else:
            seq_x = aa_emb

        seq_x = self.pos_encoder(seq_x)

        # ==== Meta embedding ====
        meta = self.meta_embedding(charge, nce)
        meta = meta.unsqueeze(1).repeat(1, L, 1)

        x = torch.cat([seq_x, meta], dim=2)

        # ==== Backbone ====
        hidden = self.transformer(x)

        # residual trick (paper exact)
        hidden = self.dropout(hidden + 0.2 * x)

        # ==== Main head ====
        out_main = self.output_head(hidden)

        # ==== Modloss ====
        if self.num_modloss > 0:
            if self.mask_modloss:
                zeros = torch.zeros(
                    B, L, self.num_modloss,
                    device=x.device
                )
                out = torch.cat([out_main, zeros], dim=2)
            else:
                modloss_x = self.modloss_transformer(x)
                modloss_x = modloss_x + hidden
                modloss_out = self.modloss_head(modloss_x)
                out = torch.cat([out_main, modloss_out], dim=2)
        else:
            out = out_main

        # remove first cleavage position
        return out[:, 1:, :]
print("hello1")

hello1


In [26]:
model = ModelMS2Transformer(
    aa_vocab_size=30,   # kiểm tra vocab size thật nếu cần
    mod_dim=0,
    num_frag_types=6,
    num_modloss_types=0,
    hidden=256
)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

seq, charge, nce, target, mask = next(iter(loader))

seq = seq.to(device)
charge = charge.to(device)
nce = nce.to(device)

pred = model(seq, None, charge, nce)

print("pred shape:", pred.shape)
print("target shape:", target.shape)

pred shape: torch.Size([8, 29, 6])
target shape: torch.Size([8, 29, 6])


In [28]:
print(pred[0])
print(target[0])

tensor([[ 0.1670,  0.1145,  0.2148, -0.1949, -0.2296, -0.0252],
        [ 0.5477,  0.0162,  0.2159,  0.0362, -0.4161, -0.3857],
        [ 0.0776, -0.2664,  0.2807,  0.1265, -0.1121, -0.2119],
        [ 0.2035, -0.3315,  0.3200,  0.0916,  0.1136, -0.1690],
        [ 0.0149, -0.4181,  0.2685, -0.1941, -0.0111, -0.2554],
        [ 0.0205, -0.3731,  0.1962, -0.0026,  0.2088, -0.1691],
        [ 0.2974, -0.1994,  0.6285, -0.1864, -0.3512, -0.1578],
        [ 0.4325, -0.3273,  0.5424, -0.0239,  0.1468, -0.2286],
        [ 0.5380,  0.4600, -0.0925, -0.2399, -0.1777, -0.3912],
        [ 0.2226,  0.1326, -0.0504,  0.1956,  0.0610, -0.5376],
        [ 0.3449,  0.2879,  0.2150,  0.2493,  0.1858, -0.2843],
        [ 0.2853,  0.0204,  0.0583,  0.1605,  0.1448, -0.1260],
        [ 0.4242,  0.1140,  0.5064,  0.1586,  0.0884, -0.1961],
        [ 0.4467, -0.0042,  0.1773,  0.1561,  0.0185, -0.2501],
        [ 0.6347,  0.2947,  0.1342,  0.1327, -0.0019, -0.0909],
        [ 0.2449,  0.1629,  0.3752,  0.1

In [29]:
print(mask[0])

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [30]:
mask = mask.unsqueeze(-1)
print(mask.shape)
print(mask[0])

torch.Size([8, 29, 1])
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [31]:
mask = mask.cuda()
target = target.cuda()

In [32]:
print(pred.device)
print(target.device)
print(mask.device)

cuda:0
cuda:0
cuda:0


In [33]:
loss = torch.abs(pred - target) * mask
loss[0]

tensor([[0.1136, 0.1145, 0.2148, 0.1949, 0.2296, 0.0252],
        [0.4776, 0.0162, 0.2159, 0.1215, 0.4161, 0.3857],
        [0.1500, 0.2664, 0.2807, 0.1265, 0.1121, 0.2119],
        [0.3630, 0.3837, 0.3200, 0.0916, 0.1136, 0.1690],
        [0.3320, 0.7842, 0.2685, 0.1941, 0.0111, 0.2554],
        [0.0824, 1.0401, 0.1962, 0.0026, 0.2088, 0.1691],
        [0.2396, 0.6917, 0.6142, 0.1864, 0.3512, 0.1578],
        [0.4123, 1.3273, 0.5424, 0.0239, 0.1468, 0.2286],
        [0.5380, 0.3504, 0.0925, 0.2399, 0.1777, 0.3912],
        [0.2226, 0.0325, 0.0504, 0.1956, 0.0610, 0.6108],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.000

In [34]:
loss.shape

torch.Size([8, 29, 6])

In [35]:
loss_per_sample = loss.sum(dim=(1,2))
loss_per_sample 

tensor([16.5434, 12.0242, 14.2847, 13.0393, 17.1522, 15.9941,  6.7395,  9.8753],
       device='cuda:0', grad_fn=<SumBackward1>)

In [36]:
valid = mask.sum(dim=(1,2))
valid

tensor([10., 10., 11., 10., 13., 13.,  6.,  8.], device='cuda:0')

In [37]:
valid.clamp(min=1)


tensor([10., 10., 11., 10., 13., 13.,  6.,  8.], device='cuda:0')

In [38]:
loss_per_sample = loss_per_sample / valid.clamp(min=1)
loss_per_sample

tensor([1.6543, 1.2024, 1.2986, 1.3039, 1.3194, 1.2303, 1.1232, 1.2344],
       device='cuda:0', grad_fn=<DivBackward0>)

In [39]:
def masked_l1_loss(pred, target, mask):

    mask = mask.unsqueeze(-1)

    loss = torch.abs(pred - target) * mask

    # sum fragment + ion
    loss_per_sample = loss.sum(dim=(1,2))

    # số phần tử hợp lệ mỗi peptide
    valid = mask.sum(dim=(1,2))

    loss_per_sample = loss_per_sample / valid.clamp(min=1)

    return loss_per_sample.mean()

In [40]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

seq, charge, nce, target, mask = next(iter(loader))

seq = seq.to(device)
charge = charge.to(device)
nce = nce.to(device)
target = target.to(device)
mask = mask.to(device)
# Chuyển tất cả -1 trong target về 0 trước khi tính Loss


pred = model(seq, None, charge, nce)

loss = masked_l1_loss(pred, target, mask)
print("initial loss:", loss.item())
print("min:", pred.min().item())
print("max:", pred.max().item())
print("target min:", target.min().item())
print("target max:", target.max().item())

initial loss: 1.3307327032089233
min: -0.7655959129333496
max: 0.9160541892051697
target min: 0.0
target max: 1.0


In [41]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for step in range(200):
    pred = model(seq, None, charge, nce)
    loss = masked_l1_loss(pred, target, mask)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(step, loss.item())

0 1.3591654300689697
20 0.46804726123809814
40 0.4121893644332886
60 0.3874812126159668
80 0.3722878098487854
100 0.35640406608581543
120 0.34995320439338684
140 0.34627729654312134
160 0.32797563076019287
180 0.32784196734428406


In [42]:
print(f"Số lượng giá trị > 0 trong Pred: {(pred > 0).sum().item()}")

Số lượng giá trị > 0 trong Pred: 921


In [43]:
print(pred[0][0])
print(target[0][0])

tensor([ 0.0746, -0.0130, -0.0048,  0.0030,  0.0132, -0.0099], device='cuda:0',
       grad_fn=<SelectBackward0>)
tensor([0.2298, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000], device='cuda:0')


In [44]:
from torch.utils.data import random_split

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [45]:
train_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,          # tạm tắt shuffle khi dùng HDF5
    num_workers=4,
    pin_memory=True,
    
)

val_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,          # tạm tắt shuffle khi dùng HDF5
    num_workers=4,
    pin_memory=True,
    
)

In [46]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(next(model.parameters()).device)

Total parameters: 3,183,333
cuda:0


In [47]:
import time
start = time.time()

for i, _ in enumerate(train_loader):
    if i == 3:
        break

print("3 batch load time:", time.time() - start)

3 batch load time: 1.7563014030456543


In [48]:
best_val = float("inf")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,      # giảm lr ×0.5
    patience=2,      # 2 epoch không cải thiện
    min_lr=1e-6,
)

In [49]:
for epoch in range(8):

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for seq, charge, nce, target, mask in train_loader:

        seq = seq.to(device)
        charge = charge.to(device)
        nce = nce.to(device)
        target = target.to(device)
        mask = mask.to(device)
       

        pred = model(seq, None, charge, nce)

        loss = masked_l1_loss(pred, target, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for seq, charge, nce, target, mask in val_loader:

            seq = seq.to(device)
            charge = charge.to(device)
            nce = nce.to(device)
            target = target.to(device)
            mask = mask.to(device)


            pred = model(seq, None, charge, nce)
           
            loss = masked_l1_loss(pred, target, mask)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    # ---- scheduler ----
    scheduler.step(val_loss)

    # ---- save best model ----
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_ms2_model.pt")
        print("Saved best model")

    lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"LR {lr:.2e}"
    )

Saved best model
Epoch 0 | Train 0.1999 | Val 0.1221 | LR 1.00e-03
Saved best model
Epoch 1 | Train 0.1305 | Val 0.1074 | LR 1.00e-03
Saved best model
Epoch 2 | Train 0.1164 | Val 0.1020 | LR 1.00e-03
Saved best model
Epoch 3 | Train 0.1094 | Val 0.0967 | LR 1.00e-03
Saved best model
Epoch 4 | Train 0.1040 | Val 0.0923 | LR 1.00e-03
Saved best model
Epoch 5 | Train 0.0999 | Val 0.0892 | LR 1.00e-03
Saved best model
Epoch 6 | Train 0.0981 | Val 0.0865 | LR 1.00e-03
Epoch 7 | Train 0.0967 | Val 0.0869 | LR 1.00e-03


In [50]:
path_test  = path + "/holdout_hcd.hdf5"
path_test

'/kaggle/input/datasets/lucasxyz/hcd-data/holdout_hcd.hdf5'

In [51]:
f = h5py.File(path_test, "r")
for key in list(f.keys()):
    print(key)


collision_energy
collision_energy_aligned
collision_energy_aligned_normed
intensities_raw
masses_pred
masses_raw
method
precursor_charge_onehot
rawfile
reverse
scan_number
score
sequence_integer


In [52]:
sequence_test = f["sequence_integer"][:]
intensity_test = f["intensities_raw"][:]
nce_test = f["collision_energy_aligned_normed"][:]
charge_test = f["precursor_charge_onehot"][:]


In [53]:
intensity_test[0]

array([ 0.24673085,  0.        , -1.        ,  0.        ,  0.        ,
       -1.        ,  0.76457227,  0.        , -1.        ,  1.        ,
        0.        , -1.        ,  0.05143985,  0.        , -1.        ,
        0.38610104,  0.        , -1.        ,  0.        ,  0.        ,
       -1.        ,  0.04143096,  0.        , -1.        ,  0.        ,
        0.        , -1.        ,  0.        ,  0.        , -1.        ,
        0.        ,  0.        , -1.        ,  0.        ,  0.        ,
       -1.        ,  0.19679859,  0.        , -1.        ,  0.        ,
        0.        , -1.        ,  0.45600307,  0.        , -1.        ,
        0.        ,  0.        , -1.        ,  0.60845343,  0.        ,
       -1.        ,  0.        ,  0.        , -1.        ,  0.56754551,
        0.31890353, -1.        ,  0.66913636,  0.        , -1.        ,
        0.06513603,  0.05947668, -1.        ,  0.        ,  0.        ,
       -1.        , -1.        , -1.        , -1.        , -1.  

In [54]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Chuyển đổi Charge One-hot sang số thực (0.1, 0.2, ...)
charge_tensor = torch.tensor(charge_test, dtype=torch.float32)
all_charge_test = (charge_tensor.argmax(dim=1) * 0.1 + 0.1).unsqueeze(1)

# 2. Chuyển các thành phần khác sang Tensor
all_seq_test = torch.tensor(sequence_test, dtype=torch.long)
all_nce_test = torch.tensor(nce_test, dtype=torch.float32)
all_intensity_test = torch.tensor(intensity_test, dtype=torch.float32)

# 3. Tạo Mask (để biết chiều dài thực của peptide)
all_mask_test = (all_seq_test != 0).float()

# 4. Đóng gói vào DataLoader
test_dataset = TensorDataset(all_seq_test, all_charge_test, all_nce_test, all_intensity_test, all_mask_test)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)



In [55]:
def calculate_sa(t, p):
    # Spectral Angle đo góc giữa 2 vector phổ
    # Càng gần 1 càng tốt
    norm_t = np.linalg.norm(t)
    norm_p = np.linalg.norm(p)
    if norm_t == 0 or norm_p == 0: return 0
    
    cos_theta = np.dot(t, p) / (norm_t * norm_p)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return 1 - (2 * np.arccos(cos_theta) / np.pi)

In [56]:
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

model.eval()
all_pccs = []
all_sas = []

print("Đang bắt đầu Evaluate tập Test...")

with torch.no_grad():
    for seq, charge, nce, raw_int, mask in tqdm(test_loader):
        # Đẩy input lên GPU
        seq_gpu = seq.to(device)
        charge_gpu = charge.to(device)
        nce_gpu = nce.to(device)
        
        # 1. Model dự đoán (Output: Batch, 29, 6)
        with torch.amp.autocast('cuda'):
            preds = model(seq_gpu, None, charge_gpu, nce_gpu)
        
        preds_np = preds.cpu().numpy()
        raw_int_np = raw_int.numpy() 
        mask_np = mask.numpy()
        
        # 2. Xử lý từng mẫu trong batch
        # 2. Xử lý từng mẫu trong batch
        for i in range(preds_np.shape[0]):
            seq_len = int(mask_np[i].sum())
            if seq_len <= 1: continue
            
            # Chuyển đổi sang format pDeep (29, 6)
            target_29_6 = prosit_to_pdeep_fast(raw_int_np[i], seq_len).numpy()

            # --- SỬA Ở ĐÂY ---
            target_29_6[target_29_6 < 0] = 0
           
            p = preds_np[i, :seq_len-1, :].flatten()
            t = target_29_6[:seq_len-1, :].flatten()
            
            p = np.maximum(p, 0) 
            
            # 3. Tính PCC và SA
            if np.std(t) > 1e-12 and np.std(p) > 1e-12:
                pcc, _ = pearsonr(t, p)
                sa = calculate_sa(t, p)
                all_pccs.append(pcc)
                all_sas.append(sa)
            else:
                # Nếu phổ trống không (toàn 0) thì cho Metrics bằng 0 
                # thay vì bỏ qua (continue) để số lượng mẫu khớp 100%
                all_pccs.append(0)
                all_sas.append(0)

print("\n" + "="*30)
print(f"Mean PCC:   {np.mean(all_pccs):.4f}")
print(f"Median PCC: {np.median(all_pccs):.4f}")
print(f"Mean SA: {np.mean(all_sas):.4f}")
print(f"Median SA: {np.median(all_sas):.4f}")


print("="*30)

Đang bắt đầu Evaluate tập Test...


100%|██████████| 1474/1474 [07:56<00:00,  3.09it/s]



Mean PCC:   0.9703
Median PCC: 0.9868
Mean SA: 0.8801
Median SA: 0.9020


In [57]:
test_t = np.array([1.0, 0.5, 0.0])
test_p = np.array([0.9, 0.4, 0.1])
print(calculate_sa(test_t, test_p))


0.9294092120087254
